[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/agentes-avanzados/03-computer-use.ipynb)

# Computer Use: Claude Controlando el Ordenador

> Aprende a usar la funcionalidad de Computer Use de Claude para automatizar
> tareas en el escritorio: navegación web, formularios y manipulación de archivos.

## Objetivos
- Entender la arquitectura de Computer Use de Anthropic
- Implementar las herramientas de ordenador (screenshot, click, type)
- Construir un agente que navega la web de forma autónoma
- Añadir salvaguardas de seguridad para uso responsable

**Nota:** Computer Use requiere una pantalla activa (real o virtual). Este notebook explica la arquitectura y proporciona el código completo. Para ejecutar en entorno real, usa la imagen Docker oficial de Anthropic.

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic pillow --quiet

## 2. Arquitectura de Computer Use

In [ ]:
print("""=== ARQUITECTURA DE COMPUTER USE ===

Computer Use permite a Claude interactuar con el ordenador como un humano:

FLUJO DE EJECUCIÓN:
  1. Usuario da instrucción en lenguaje natural
  2. Claude solicita una captura de pantalla (tool: screenshot)
  3. Claude analiza la imagen y decide qué acción tomar
  4. Claude ejecuta la acción (click, type, scroll, key)
  5. Repetir desde paso 2 hasta completar la tarea

HERRAMIENTAS DISPONIBLES:
  📷 computer.screenshot() → captura la pantalla
  🖱️ computer.left_click(x, y) → clic izquierdo
  🖱️ computer.right_click(x, y) → clic derecho
  ⌨️ computer.type(text) → escribir texto
  ⌨️ computer.key(key) → tecla especial (Enter, Tab, Escape)
  🖱️ computer.scroll(x, y, direction, amount) → scroll
  🖱️ computer.mouse_move(x, y) → mover ratón

MODELO REQUERIDO:
  claude-sonnet-4-6 (o superior)
  Versión de API: 2023-06-01 con beta: computer-use-2024-10-22

CASOS DE USO REALES:
  ✓ Rellenar formularios web complejos
  ✓ Extraer datos de aplicaciones de escritorio
  ✓ Automatizar tareas repetitivas en ERP/CRM
  ✓ Testing de interfaces de usuario
  ✓ Navegación web autónoma
""")

## 3. Implementar las herramientas de Computer Use

In [ ]:
import anthropic
import base64
import json
from typing import Optional

# Definición de las herramientas para Computer Use
COMPUTER_TOOL = {
    "type": "computer_20241022",
    "name": "computer",
    "display_width_px": 1280,
    "display_height_px": 800,
    "display_number": 1,
}

# Simulador de acciones (en producción: usar pyautogui, xdotool o playwright)
class SimuladorOrdenador:
    """Simula acciones del ordenador para demostración sin pantalla real."""

    def __init__(self, ancho: int = 1280, alto: int = 800):
        self.ancho = ancho
        self.alto = alto
        self.posicion_raton = (640, 400)
        self.historial_acciones = []

    def screenshot(self) -> str:
        """Captura de pantalla (simulada: imagen gris con texto)."""
        try:
            from PIL import Image, ImageDraw
            img = Image.new("RGB", (self.ancho, self.alto), color=(240, 240, 240))
            draw = ImageDraw.Draw(img)
            draw.rectangle([0, 0, self.ancho, 40], fill=(70, 130, 180))
            draw.text((10, 10), "Navegador simulado — http://ejemplo.com", fill="white")
            draw.rectangle([10, 60, self.ancho-10, 120], fill="white", outline="gray")
            draw.text((20, 75), "Campo de búsqueda", fill="gray")
            draw.rectangle([10, 140, 300, 180], fill=(70, 130, 180))
            draw.text((80, 153), "Buscar", fill="white")

            import io
            buffer = io.BytesIO()
            img.save(buffer, format="PNG")
            return base64.b64encode(buffer.getvalue()).decode()
        except ImportError:
            # Sin PIL: devolver imagen 1x1 blanca
            return "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mNkYPhfDwAChwGA60e6kgAAAABJRU5ErkJggg=="

    def ejecutar_accion(self, accion: dict) -> dict:
        """Ejecuta una acción y devuelve el resultado."""
        tipo = accion.get("action")
        self.historial_acciones.append(accion)

        if tipo == "screenshot":
            imagen = self.screenshot()
            return {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": imagen}}

        elif tipo in ["left_click", "right_click", "double_click", "mouse_move"]:
            x, y = accion.get("coordinate", [0, 0])
            self.posicion_raton = (x, y)
            print(f"  [SIM] {tipo} en ({x}, {y})")
            return {"type": "tool_result", "content": f"Click ejecutado en ({x}, {y})"}

        elif tipo == "type":
            texto = accion.get("text", "")
            print(f"  [SIM] type: '{texto}'")
            return {"type": "tool_result", "content": f"Texto escrito: {texto}"}

        elif tipo == "key":
            tecla = accion.get("text", "")
            print(f"  [SIM] key: {tecla}")
            return {"type": "tool_result", "content": f"Tecla presionada: {tecla}"}

        elif tipo == "scroll":
            direccion = accion.get("direction", "down")
            print(f"  [SIM] scroll {direccion}")
            return {"type": "tool_result", "content": f"Scroll {direccion}"}

        return {"type": "tool_result", "content": "Acción desconocida"}

sim = SimuladorOrdenador()
print("✓ Simulador de ordenador inicializado")
print(f"  Resolución: {sim.ancho}×{sim.alto}px")
print("  Nota: En producción, sustituir SimuladorOrdenador por pyautogui/playwright")

## 4. Agente de Computer Use con bucle de herramientas

In [ ]:
client = anthropic.Anthropic()

# Salvaguardas de seguridad
SITIOS_BLOQUEADOS = ["bank", "paypal", "admin", "ssh", "terminal"]
MAX_ITERACIONES = 15

def verificar_seguridad(accion: dict) -> bool:
    """Verifica que la acción es segura antes de ejecutarla."""
    if accion.get("action") == "type":
        texto = accion.get("text", "").lower()
        palabras_peligrosas = ["rm -rf", "drop table", "delete from", "format c"]
        if any(p in texto for p in palabras_peligrosas):
            print(f"  ⛔ BLOQUEADO: texto peligroso detectado: {texto[:50]}")
            return False
    return True

def agente_computer_use(tarea: str, max_iteraciones: int = MAX_ITERACIONES) -> dict:
    """Ejecuta un agente de Computer Use para completar una tarea."""
    print(f"\n=== AGENTE COMPUTER USE ===")
    print(f"Tarea: {tarea}")
    print("-" * 50)

    mensajes = [{"role": "user", "content": tarea}]
    iteraciones = 0
    acciones_ejecutadas = []

    while iteraciones < max_iteraciones:
        iteraciones += 1
        print(f"\nIteración {iteraciones}/{max_iteraciones}")

        # Llamar a Claude con Computer Use
        resp = client.beta.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1024,
            tools=[COMPUTER_TOOL],
            messages=mensajes,
            betas=["computer-use-2024-10-22"],
        )

        print(f"  Stop reason: {resp.stop_reason}")

        # Procesar respuesta
        resultados_herramientas = []
        texto_agente = ""

        for bloque in resp.content:
            if bloque.type == "text":
                texto_agente = bloque.text
                print(f"  Claude: {bloque.text[:100]}")

            elif bloque.type == "tool_use" and bloque.name == "computer":
                accion = bloque.input
                print(f"  Acción solicitada: {accion.get('action')}")

                if verificar_seguridad(accion):
                    resultado = sim.ejecutar_accion(accion)
                    acciones_ejecutadas.append(accion)
                else:
                    resultado = {"type": "tool_result",
                                "content": "Acción bloqueada por salvaguardas de seguridad"}

                resultados_herramientas.append({
                    "type": "tool_result",
                    "tool_use_id": bloque.id,
                    "content": [resultado] if isinstance(resultado, dict) and resultado.get("type") == "image" else resultado.get("content", ""),
                })

        # Actualizar historial de mensajes
        mensajes.append({"role": "assistant", "content": resp.content})

        if resp.stop_reason == "end_turn":
            print("\n✓ Tarea completada")
            break

        if resultados_herramientas:
            mensajes.append({"role": "user", "content": resultados_herramientas})

    return {
        "completado": resp.stop_reason == "end_turn",
        "iteraciones": iteraciones,
        "acciones_ejecutadas": len(acciones_ejecutadas),
        "conclusion": texto_agente,
    }

# Ejecutar el agente con una tarea de demostración
resultado = agente_computer_use(
    "Toma una captura de pantalla y describe qué ves. Luego haz clic en el campo de búsqueda."
)
print(f"\nResumen: {resultado['acciones_ejecutadas']} acciones en {resultado['iteraciones']} iteraciones")

## 5. Configuración Docker para entorno real

In [ ]:
DOCKER_COMPOSE_CU = """
# Entorno oficial de Anthropic para Computer Use
# Incluye: Chrome, VNC, noVNC para visualización

version: '3.8'
services:
  computer-use-demo:
    image: ghcr.io/anthropics/anthropic-quickstarts:computer-use-demo-latest
    environment:
      - ANTHROPIC_API_KEY=${ANTHROPIC_API_KEY}
    ports:
      - "5900:5900"  # VNC
      - "8501:8501"  # Streamlit UI
      - "6080:6080"  # noVNC (acceso web)
      - "8080:8080"  # API interna
    shm_size: 1gb  # Necesario para Chrome
"""

CODIGO_PRODUCCION = """
# En producción: usar pyautogui para control real del escritorio
import pyautogui
import subprocess
import base64
from PIL import ImageGrab
import io

class OrdenadorReal:
    def screenshot(self) -> str:
        img = ImageGrab.grab()
        buffer = io.BytesIO()
        img.save(buffer, format='PNG')
        return base64.b64encode(buffer.getvalue()).decode()

    def left_click(self, x, y):
        pyautogui.click(x, y)

    def type_text(self, text):
        pyautogui.typewrite(text, interval=0.05)

    def key(self, key_name):
        pyautogui.press(key_name)

    def scroll(self, x, y, direction='down', amount=3):
        pyautogui.scroll(-amount if direction == 'down' else amount, x=x, y=y)

# Para navegación web: usar Playwright (más robusto que pyautogui)
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=False)
    page = browser.new_page()
    page.goto('https://example.com')
    # Claude puede hacer screenshot del viewport
    screenshot_bytes = page.screenshot()
"""

print("=== OPCIONES PARA ENTORNO REAL ===")
print("""
Opción 1 — Imagen oficial Anthropic (recomendada):
  docker run -e ANTHROPIC_API_KEY=$KEY \\
    -p 5900:5900 -p 8501:8501 -p 6080:6080 \\
    ghcr.io/anthropics/anthropic-quickstarts:computer-use-demo-latest

  Acceder a: http://localhost:6080 (noVNC) o http://localhost:8501 (Streamlit)

Opción 2 — Control nativo (Linux/Mac/Windows):
  pip install pyautogui playwright pillow
  playwright install chromium
  Sustituir SimuladorOrdenador por OrdenadorReal (código arriba)

Opción 3 — Playwright para web (sin escritorio):
  pip install playwright
  playwright install chromium
  Usar page.screenshot() → base64 → Claude
"""
)

print("=== SALVAGUARDAS RECOMENDADAS EN PRODUCCIÓN ===")
print("""
  ✓ Sandbox de red (sin acceso a internet salvo sitios en whitelist)
  ✓ Captura de pantalla antes de cada acción (auditoría)
  ✓ Aprobación humana para acciones irreversibles (enviar, borrar, comprar)
  ✓ Timeout máximo de sesión (evitar bucles infinitos)
  ✓ Lista negra de aplicaciones (banking, terminal, admin)
  ✓ Modo de solo lectura por defecto
""")